# Global E-Commerce Sales Data Cleaning
Synapse PySpark notebook for cleaning and transforming the dataset.

## Automated data-quality checks

This notebook validates required source columns, records exact duplicates, identifies missing/invalid values and failed date/numeric conversions, derives revenue, and checks the saved Parquet output after writing. Invalid rows are written with `_DQ_REASONS`; a run-level CSV summary is written to the configured quality path. Negative prices/quantities are reported for review rather than automatically rejected because returns and credits may be legitimate.

**Important:** update the ADLS Gen2 paths and `required_columns` configuration for your environment before running.

In [ ]:
# Synapse PySpark Notebook Code
# Global E-Commerce Sales Data Cleaning with automated quality checks.

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

# --- Environment configuration ---
# Update these paths for your own ADLS Gen2 container before running the notebook.
input_path = "abfss://shinyblobcontainer@shinyshastoreacc.dfs.core.windows.net/global E commerce sales dataset/*.csv"
output_path = "abfss://shinyblobcontainer@shinyshastoreacc.dfs.core.windows.net/cleaned/ecommerce_cleaned"
powerbi_output_path = "abfss://shinyblobcontainer@shinyshastoreacc.dfs.core.windows.net/gold/ecommerce_sales_powerbi"
rejected_root_path = "abfss://shinyblobcontainer@shinyshastoreacc.dfs.core.windows.net/rejected/ecommerce_sales"
quality_report_path = "abfss://shinyblobcontainer@shinyshastoreacc.dfs.core.windows.net/quality/ecommerce_sales"

# These columns are used by the supplied revenue/region/category/product SQL analytics.
# Change this list if your source uses different business column names.
required_columns = ["Price", "Quantity", "Region", "Category", "Product"]
date_column = "Transaction Date"

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_reject_path = f"{rejected_root_path}/run_id={run_id}"

# Read source values as strings so malformed numeric values can be identified
# before type conversion instead of being silently coerced during schema inference.
raw_df = (
    spark.read.option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .csv(input_path)
)

source_columns = raw_df.columns
missing_required_columns = [name for name in required_columns if name not in source_columns]

print("Run ID:", run_id)
print("Source columns:", source_columns)

# Fail fast when an expected field is absent. The error explicitly lists the
# missing names, so the user can correct the source or required_columns setting.
if missing_required_columns:
    raise ValueError(
        "Required source columns are missing: "
        + ", ".join(missing_required_columns)
        + ". Check the CSV header and required_columns configuration."
    )

source_rows = raw_df.count()
deduplicated_df = raw_df.dropDuplicates()

# Extra copies of exact full-row duplicates are preserved separately for audit.
# exceptAll retains duplicate multiplicity, so this returns the excess copies.
duplicate_rows = raw_df.exceptAll(deduplicated_df)

price_text = F.trim(F.col("Price"))
quantity_text = F.trim(F.col("Quantity"))
price_value = price_text.cast("double")
quantity_value = quantity_text.cast("integer")

price_missing = F.col("Price").isNull() | (price_text == "")
quantity_missing = F.col("Quantity").isNull() | (quantity_text == "")

price_non_finite = (
    F.isnan(price_value) | (F.abs(price_value) == F.lit(float("inf")))
)
price_invalid = (~price_missing) & (price_value.isNull() | price_non_finite)
quantity_invalid = (~quantity_missing) & quantity_value.isNull()

work_df = (
    deduplicated_df.withColumn("_DQ_PRICE", price_value)
    .withColumn("_DQ_QUANTITY", quantity_value)
)

reason_expressions = [
    F.when(price_missing, F.lit("MISSING_PRICE")),
    F.when(price_invalid, F.lit("INVALID_PRICE")),
    F.when(quantity_missing, F.lit("MISSING_QUANTITY")),
    F.when(quantity_invalid, F.lit("INVALID_QUANTITY")),
]

# Dimension fields required by the supplied SQL analytics must not be blank.
dimension_columns = ["Region", "Category", "Product"]
for column_name in dimension_columns:
    trimmed_value = F.trim(F.col(column_name))
    is_missing = F.col(column_name).isNull() | (trimmed_value == "")
    reason_code = "MISSING_" + column_name.upper().replace(" ", "_")
    reason_expressions.append(F.when(is_missing, F.lit(reason_code)))

if date_column in source_columns:
    date_text = F.trim(F.col(date_column))
    parsed_date = F.to_date(date_text)
    date_missing = F.col(date_column).isNull() | (date_text == "")
    date_invalid = (~date_missing) & parsed_date.isNull()

    work_df = work_df.withColumn("_DQ_PARSED_DATE", parsed_date)
    reason_expressions.extend(
        [
            F.when(date_missing, F.lit("MISSING_TRANSACTION_DATE")),
            F.when(date_invalid, F.lit("INVALID_TRANSACTION_DATE")),
        ]
    )
else:
    print(
        "WARNING: Optional date column is absent; date-derived fields will not be created:",
        date_column,
    )

work_df = work_df.withColumn(
    "_DQ_REVENUE", F.col("_DQ_PRICE") * F.col("_DQ_QUANTITY").cast("double")
)

# Only classify revenue as invalid when both numeric inputs themselves are valid.
numeric_inputs_valid = (
    (~price_missing)
    & (~price_invalid)
    & (~quantity_missing)
    & (~quantity_invalid)
)
revenue_value = F.col("_DQ_REVENUE")
revenue_non_finite = (
    F.isnan(revenue_value) | (F.abs(revenue_value) == F.lit(float("inf")))
)
invalid_revenue = numeric_inputs_valid & (
    revenue_value.isNull() | revenue_non_finite
)
reason_expressions.append(F.when(invalid_revenue, F.lit("INVALID_REVENUE")))

work_df = work_df.withColumn(
    "_DQ_REASONS", F.concat_ws("; ", *reason_expressions)
)

reason_codes = [
    "MISSING_PRICE",
    "INVALID_PRICE",
    "MISSING_QUANTITY",
    "INVALID_QUANTITY",
    "MISSING_REGION",
    "MISSING_CATEGORY",
    "MISSING_PRODUCT",
    "MISSING_TRANSACTION_DATE",
    "INVALID_TRANSACTION_DATE",
    "INVALID_REVENUE",
]

# Aggregate all reason counts in one Spark aggregation.
summary_expressions = [
    F.count(F.lit(1)).alias("deduplicated_rows"),
    F.sum(
        F.when(F.length(F.col("_DQ_REASONS")) > 0, 1).otherwise(0)
    ).alias("rejected_rows"),
]
for reason_code in reason_codes:
    summary_expressions.append(
        F.sum(
            F.when(F.instr(F.col("_DQ_REASONS"), reason_code) > 0, 1).otherwise(0)
        ).alias(reason_code.lower())
    )

summary_expressions.extend(
    [
        F.sum(F.when((F.length(F.col("_DQ_REASONS")) == 0) & (F.col("_DQ_PRICE") < 0), 1).otherwise(0)).alias(
            "negative_price_rows"
        ),
        F.sum(F.when((F.length(F.col("_DQ_REASONS")) == 0) & (F.col("_DQ_QUANTITY") < 0), 1).otherwise(0)).alias(
            "negative_quantity_rows"
        ),
    ]
)
summary = work_df.agg(*summary_expressions).first()


def metric_count(name):
    value = summary[name]
    return int(value) if value is not None else 0


deduplicated_rows = metric_count("deduplicated_rows")
exact_duplicate_rows = source_rows - deduplicated_rows
rejected_rows = metric_count("rejected_rows")
accepted_rows = deduplicated_rows - rejected_rows

# Write rejected rows, including the source fields and a machine-readable reason.
# Duplicate copies and invalid records are separated for easier investigation.
if exact_duplicate_rows > 0:
    (
        duplicate_rows.withColumn("_DQ_REASONS", F.lit("EXACT_DUPLICATE"))
        .write.mode("overwrite")
        .parquet(f"{run_reject_path}/exact_duplicates")
    )

rejected_df = work_df.filter(F.length(F.col("_DQ_REASONS")) > 0)
rejected_output_df = rejected_df.drop(
    "_DQ_PRICE", "_DQ_QUANTITY", "_DQ_PARSED_DATE", "_DQ_REVENUE"
)
if rejected_rows > 0:
    (
        rejected_output_df.write.mode("overwrite")
        .parquet(f"{run_reject_path}/invalid_records")
    )

# Accept only rows that pass the configured data-quality rules.
accepted_df = work_df.filter(F.length(F.col("_DQ_REASONS")) == 0)
cleaned_df = (
    accepted_df.withColumn("Price", F.col("_DQ_PRICE"))
    .withColumn("Quantity", F.col("_DQ_QUANTITY"))
    .withColumn("Revenue", F.col("_DQ_REVENUE"))
)

if date_column in source_columns:
    cleaned_df = (
        cleaned_df.withColumn("Transaction_Date", F.col("_DQ_PARSED_DATE"))
        .withColumn("Transaction_Month", F.month(F.col("_DQ_PARSED_DATE")))
        .withColumn("Transaction_Year", F.year(F.col("_DQ_PARSED_DATE")))
    )

cleaned_df = cleaned_df.drop(
    "_DQ_PRICE",
    "_DQ_QUANTITY",
    "_DQ_PARSED_DATE",
    "_DQ_REVENUE",
    "_DQ_REASONS",
)

print("Input rows:", source_rows)
print("Exact duplicate rows routed to reject output:", exact_duplicate_rows)
print("Rows rejected by quality rules:", rejected_rows)
print("Rows accepted for output:", accepted_rows)
print("Negative prices retained for business review:", metric_count("negative_price_rows"))
print(
    "Negative quantities retained for business review:",
    metric_count("negative_quantity_rows"),
)

# Keep the same cleaned-output path used by the SQL analytics scripts.
cleaned_df.write.mode("overwrite").parquet(output_path)

# Preserve the optional secondary output from the original notebook.
# coalesce(1) is suitable only for small datasets; use partitioned output at scale.
cleaned_df.coalesce(1).write.mode("overwrite").parquet(powerbi_output_path)

# Read the saved Parquet back to verify output count and revenue arithmetic.
readback_df = spark.read.parquet(output_path)
output_rows = readback_df.count()
expected_output_columns = [
    "Price",
    "Quantity",
    "Region",
    "Category",
    "Product",
    "Revenue",
]
missing_output_columns = [
    name for name in expected_output_columns if name not in readback_df.columns
]

if all(name in readback_df.columns for name in ["Price", "Quantity", "Revenue"]):
    expected_revenue = F.col("Price").cast("double") * F.col("Quantity").cast("double")
    saved_revenue = F.col("Revenue").cast("double")
    revenue_mismatch = (
        saved_revenue.isNull()
        | F.isnan(saved_revenue)
        | (F.abs(saved_revenue) == F.lit(float("inf")))
        | (F.abs(saved_revenue - expected_revenue) > F.lit(0.000001))
    )
    revenue_readback_issue_rows = (
        readback_df.agg(
            F.sum(F.when(revenue_mismatch, 1).otherwise(0)).alias("issues")
        ).first()["issues"]
        or 0
    )
    revenue_readback_issue_rows = int(revenue_readback_issue_rows)
else:
    revenue_readback_issue_rows = output_rows

# Save a compact, run-level CSV report. The folder is overwritten on each run,
# so this path always points to the latest summary. Detailed rejects are versioned.
def status_for_positive_count(count):
    return "WARN" if count > 0 else "PASS"


quality_metrics = [
    (run_id, "required_columns_present", "PASS", len(required_columns), ", ".join(required_columns)),
    (run_id, "input_rows", "INFO", source_rows, input_path),
    (run_id, "deduplicated_rows", "INFO", deduplicated_rows, "Rows remaining after exact full-row deduplication."),
    (run_id, "exact_duplicate_rows", status_for_positive_count(exact_duplicate_rows), exact_duplicate_rows, f"Review under {run_reject_path}/exact_duplicates when count is greater than zero."),
    (run_id, "rows_rejected_by_quality_rules", status_for_positive_count(rejected_rows), rejected_rows, f"Reason-coded rows are saved under {run_reject_path}/invalid_records when count is greater than zero."),
    (run_id, "accepted_rows", "INFO", accepted_rows, "Rows expected in the cleaned output."),
    (run_id, "output_rows_after_parquet_readback", "PASS" if output_rows == accepted_rows else "FAIL", output_rows, f"Expected accepted row count: {accepted_rows}."),
    (run_id, "missing_output_columns", "PASS" if not missing_output_columns else "FAIL", len(missing_output_columns), ", ".join(missing_output_columns) if missing_output_columns else "All expected analytics columns exist."),
    (run_id, "revenue_readback_mismatch_rows", "PASS" if revenue_readback_issue_rows == 0 else "FAIL", revenue_readback_issue_rows, "Checks saved Revenue against Price multiplied by Quantity."),
    (run_id, "negative_price_rows_retained_for_review", "REVIEW" if metric_count("negative_price_rows") > 0 else "PASS", metric_count("negative_price_rows"), "Not rejected automatically because refunds/credits may be valid."),
    (run_id, "negative_quantity_rows_retained_for_review", "REVIEW" if metric_count("negative_quantity_rows") > 0 else "PASS", metric_count("negative_quantity_rows"), "Not rejected automatically because returns may be valid."),
]

for reason_code in reason_codes:
    count = metric_count(reason_code.lower())
    quality_metrics.append(
        (
            run_id,
            reason_code.lower() + "_rows",
            status_for_positive_count(count),
            count,
            "Rows can have more than one reason code; reason counts are not mutually exclusive.",
        )
    )

quality_schema = T.StructType(
    [
        T.StructField("run_id", T.StringType(), False),
        T.StructField("check_name", T.StringType(), False),
        T.StructField("status", T.StringType(), False),
        T.StructField("count", T.LongType(), False),
        T.StructField("details", T.StringType(), True),
    ]
)
quality_report_df = spark.createDataFrame(quality_metrics, schema=quality_schema)
(
    quality_report_df.coalesce(1)
    .write.mode("overwrite")
    .option("header", "true")
    .csv(quality_report_path)
)

display(quality_report_df.orderBy("check_name"))

# Fail the run only after the audit report has been persisted, so failed output
# validation still leaves a report to diagnose.
fatal_checks = []
if output_rows != accepted_rows:
    fatal_checks.append(
        f"Parquet readback row count {output_rows} does not match accepted rows {accepted_rows}"
    )
if missing_output_columns:
    fatal_checks.append("Missing output columns: " + ", ".join(missing_output_columns))
if revenue_readback_issue_rows > 0:
    fatal_checks.append(
        f"Saved revenue failed consistency checks for {revenue_readback_issue_rows} rows"
    )
if fatal_checks:
    raise RuntimeError("Output validation failed: " + "; ".join(fatal_checks))

print("Data quality validation completed.")
print("Cleaned Parquet output:", output_path)
print("Latest quality report:", quality_report_path)
print("Run-specific rejected rows:", run_reject_path)
